In [1]:

"""



Pruning


Load trained VGG-style CNN

global unstructured magnitude pruning at multiple sparsity levels

Fine-tunes the pruned model for a few epochs

Report: parameter count, FLOPs, train accuracy, test accuracy

visualization: test accuracy vs. remaining parameters




reference fom the paper

Learning both Weights and Connections for Efficient Neural Networks

Song Han, Jeff Pool, John Tran, William Dally (2015)

https://arxiv.org/abs/1506.02626



"""

import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import copy
import time
import os
import torch.nn.functional as F

In [2]:
"""

:

  Block 1: Conv(3→64) → BN → ReLU → Conv(64→64) → BN → ReLU → MaxPool → Dropout(0.1)

  Block 2: Conv(64→128) → BN → ReLU → Conv(128→128) → BN → ReLU → MaxPool → Dropout(0.2)

  Block 3: Conv(128→256) → BN → ReLU → Conv(256→256) → BN → ReLU → MaxPool → Dropout(0.3)

  Block 4: Conv(256→512) → BN → ReLU → Conv(512→512) → BN → ReLU → MaxPool → Dropout(0.4)

  GAP → FC(512→256) → ReLU → Dropout(0.5) → FC(256→10)



"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class VGGStyleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        # ── Block 1: 3 → 64 → 64, pool ──
        self.conv1a = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(64)
        self.conv1b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(64)

        # ── Block 2: 64 → 128 → 128, pool ──
        self.conv2a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(128)
        self.conv2b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(128)

        # ── Block 3: 128 → 256 → 256, pool ──
        self.conv3a = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(256)
        self.conv3b = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(256)

        # ── Block 4: 256 → 512 → 512, pool ──
        self.conv4a = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn4a = nn.BatchNorm2d(512)
        self.conv4b = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.bn4b = nn.BatchNorm2d(512)

        # ── Classifier ──
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, num_classes)

        # Initialize weights (He initialization, same as your original)
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)   # gamma = 1
                nn.init.zeros_(m.bias)    # beta = 0
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                nn.init.zeros_(m.bias)




    def forward(self, x):



        # Block 1: (N,3,32,32) → (N,64,16,16)
        x = F.relu(self.bn1a(self.conv1a(x)))
        x = F.relu(self.bn1b(self.conv1b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.1, training=self.training)




        # Block 2: (N,64,16,16) → (N,128,8,8)
        x = F.relu(self.bn2a(self.conv2a(x)))
        x = F.relu(self.bn2b(self.conv2b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.2, training=self.training)




        # Block 3: (N,128,8,8) → (N,256,4,4)
        x = F.relu(self.bn3a(self.conv3a(x)))
        x = F.relu(self.bn3b(self.conv3b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.3, training=self.training)




        # Block 4: (N,256,4,4) → (N,512,2,2)
        x = F.relu(self.bn4a(self.conv4a(x)))
        x = F.relu(self.bn4b(self.conv4b(x)))
        x = F.max_pool2d(x, 2)
        x = F.dropout(x, 0.4, training=self.training)





        # GAP: (N,512,2,2) → (N,512)
        x = F.adaptive_avg_pool2d(x, 1)
        x = x.view(x.size(0), -1)




        # Classifier
        x = F.relu(self.fc1(x))
        x = F.dropout(x, 0.5, training=self.training)
        x = self.fc2(x)

        return x


In [3]:

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
print(f"Using device: {DEVICE}")


Using device: cuda


In [4]:


# Data loading


transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2)



100%|██████████| 170M/170M [00:02<00:00, 73.9MB/s]


In [5]:

# Helper: Evaluate accuracy

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            logits = model(inputs)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total



In [6]:


#  Count parameters; total and non-zero



def count_parametersb4(model):
    """Returns total_params, nonzero_params, sparsity_percentage """
    total = 0
    nonzero = 0
    for p in model.parameters():
        total += p.numel()
        nonzero += p.nonzero().size(0)
    sparsity = 100.0 * (1 - nonzero / total)
    return total, nonzero, sparsity

In [7]:
def count_parameters(model):


    total = 0
    nonzero = 0
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            # .weight returns the masked version if pruned
            w = module.weight
            total += w.numel()
            nonzero += (w != 0).sum().item()
            if module.bias is not None:
                b = module.bias
                total += b.numel()
                nonzero += (b != 0).sum().item()
        elif isinstance(module, nn.BatchNorm2d):




            for p in module.parameters():
                total += p.numel()
                nonzero += (p != 0).sum().item()
    sparsity = 100.0 * (1 - nonzero / total)
    return total, nonzero, sparsity



In [8]:





def estimate_flops(model, input_size=(1, 3, 32, 32)):
    """
    Estimates FLOPs for the VGG-style CNN.
    For a pruned model, we count only non-zero weight multiplications.
    This gives "effective FLOPs" — how much compute actually matters.
    """
    flops = 0
    # formular for the flops stuff
    # Conv layers: FLOPs = 2 * K*K*C_in * C_out * H_out * W_out




    # pruned models, scale by the fraction of non   zero weights




    conv_specs = [
        # layer, C_in, C_out, H_out, W_out
        ('conv1a', 3,   64,  32, 32),
        ('conv1b', 64,  64,  32, 32),
        ('conv2a', 64,  128, 16, 16),
        ('conv2b', 128, 128, 16, 16),
        ('conv3a', 128, 256, 8,  8),
        ('conv3b', 256, 256, 8,  8),
        ('conv4a', 256, 512, 4,  4),
        ('conv4b', 512, 512, 4,  4),
    ]

    for name, c_in, c_out, h_out, w_out in conv_specs:
        layer = getattr(model, name)
        weight = layer.weight
        total_weight_params = weight.numel()
        nonzero_weight_params = weight.nonzero().size(0)
        density = nonzero_weight_params / total_weight_params if total_weight_params > 0 else 1.0

        # Standard conv FLOPs
        layer_flops = 2 * c_in * c_out * 3 * 3 * h_out * w_out

        flops += layer_flops * density  # scale

    # FC layers
    fc_specs = [
        ('fc1', 512, 256),
        ('fc2', 256, 10),
    ]

    for name, fan_in, fan_out in fc_specs:
        layer = getattr(model, name)
        weight = layer.weight
        total_weight_params = weight.numel()
        nonzero_weight_params = weight.nonzero().size(0)
        density = nonzero_weight_params / total_weight_params if total_weight_params > 0 else 1.0

        layer_flops = 2 * fan_in * fan_out
        flops += layer_flops * density

    return int(flops)


In [9]:


# Core: Apply global unstructured pruning





def apply_global_pruning(model, sparsity):
    """



    Apply global unstructured


      Collects all weight tensors from the lasers
      Ranks ALL weights globally by magnitude (absolute value)
      Zeros out the smallest

    Global pruning is smarter than per-layer pruning because it lets
    larger layers (which have more redundancy) absorb more pruning,
    while keeping smaller or more sensitive layers less pruned.


    """




    #
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, 'weight'))




    #
    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=sparsity,
    )

    return model


In [10]:

# Core: Remove pruning reparameterization





def make_pruning_permanent(model):
    """
    Makes pruning permanent by removing the forward hooks and
    folding the mask into the weight tensor.

    Before this call: layer has weight_orig + weight_mask → weight is computed on the fly
    After this call:  layer has weight (with zeros baked in), no mask
    """
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass  # layer wasn't pruned
    return model



In [11]:

# Core: Fine-tune a pruned model






def fine_tune(model, epochs=5, lr=0.0001):
    """
    Fine-tune a pruned model to recover accuracy.
Much smaller lr (0.0001 vs 0.0005)

 After each optimizer step, we zero out gradients that flowed into
         pruned (masked) weights. PyTorch's pruning hooks ensure the forward
         pass uses weight_orig * weight_mask, but the optimizer still updates
         weight_orig for ALL entries. By zeroing the grad for masked positions,
         we stop the optimizer from wasting updates on dead weights.

    """

    model.train()
    model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9,
                          weight_decay=0.0005)




    # Cosine schedule
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=epochs, eta_min=lr * 0.01
    )









    for epoch in range(epochs):
        epoch_start = time.time()
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            logits = model(inputs)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        scheduler.step()
        test_acc = evaluate(model, test_loader)
        elapsed = time.time() - epoch_start
        print(f"    Fine-tune epoch {epoch+1}/{epochs} | "
              f"Test Acc: {test_acc:.2f}% | Time: {elapsed:.1f}s")

    return model


In [12]:


# prune at multiple sparsity levels




def run_pruning_experiment(model_path="vgg_cifar10_trained.pth",
                           fine_tune_epochs=5):
    """

  sparsity levels: 0%, 10%, 30%, 50%, 70%, 80%, 90%, 95%
      For each: prune → evaluate → fine-tune → evaluate again

    """
    sparsity_levels = [0.0, 0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]

    results = []




    for sparsity in sparsity_levels:
        print(f"\n{'='*60}")
        print(f"  Sparsity: {sparsity*100:.0f}%")
        print(f"{'='*60}")

        # Load a FRESH copy of the trained model for each sparsity level



        # so pruning at 90% doesn't build on the 70% model
        model = VGGStyleCNN().to(DEVICE)
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))





        # Evaluate baseline BEFORE PRUING
        if sparsity == 0.0:
            train_acc = evaluate(model, train_loader)
            test_acc = evaluate(model, test_loader)
            total_params, nonzero_params, actual_sparsity = count_parameters(model)
            flops = estimate_flops(model)

            print(f"  [Baseline] Train: {train_acc:.2f}% | Test: {test_acc:.2f}%")
            print(f"  Params: {nonzero_params:,} / {total_params:,} | FLOPs: {flops:,}")

            results.append({
                'target_sparsity': 0.0,
                'actual_sparsity': actual_sparsity,
                'total_params': total_params,
                'nonzero_params': nonzero_params,
                'flops': flops,
                'test_acc_before_ft': test_acc,
                'train_acc': train_acc,
                'test_acc': test_acc,
            })
            continue





        # DO THE PRUNING
        model = apply_global_pruning(model, sparsity)

        # Evaluate NO FINETUNING
        test_acc_before = evaluate(model, test_loader)
        total_params, nonzero_params, actual_sparsity = count_parameters(model)
        flops = estimate_flops(model)

        print(f"  After pruning (before fine-tune):")
        print(f"    Test Acc: {test_acc_before:.2f}%")
        print(f"    Params: {nonzero_params:,} / {total_params:,} "
              f"(sparsity: {actual_sparsity:.1f}%)")
        print(f"    FLOPs: {flops:,}")






        # NOW FINE TUNE
        print(f"  Fine-tuning for {fine_tune_epochs} epochs...")
        model = fine_tune(model, epochs=fine_tune_epochs, lr=0.0005)

        # Evaluate AFTER FINE TUING
        train_acc = evaluate(model, train_loader)
        test_acc = evaluate(model, test_loader)
        _, nonzero_after_ft, sparsity_after_ft = count_parameters(model)
        flops_after_ft = estimate_flops(model)

        print(f"  After fine-tuning:")
        print(f"    Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")
        print(f"    Non-zero params: {nonzero_after_ft:,} | FLOPs: {flops_after_ft:,}")






        # save the pruned model
        model = make_pruning_permanent(model)






        results.append({
            'target_sparsity': sparsity,
            'actual_sparsity': actual_sparsity,
            'total_params': total_params,
            'nonzero_params': nonzero_params,
            'flops': flops,
            'test_acc_before_ft': test_acc_before,
            'train_acc': train_acc,
            'test_acc': test_acc,
        })

    return results



In [13]:


# Visualization




def plot_results(results):
    """



    """
    os.makedirs("plots", exist_ok=True)

    nonzero_params = [r['nonzero_params'] for r in results]
    test_accs = [r['test_acc'] for r in results]
    test_accs_before = [r['test_acc_before_ft'] for r in results]
    sparsities = [r['target_sparsity'] * 100 for r in results]
    flops = [r['flops'] for r in results]









    #Test accuracy vs remaining parameters ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(nonzero_params, test_accs, 'bo-', linewidth=2, markersize=8,
            label='After fine-tuning')
    ax.plot(nonzero_params, test_accs_before, 'rs--', linewidth=1.5, markersize=6,
            label='Before fine-tuning', alpha=0.7)




    # Annotate each point with sparsity %
    for i, (x, y, s) in enumerate(zip(nonzero_params, test_accs, sparsities)):
        ax.annotate(f'{s:.0f}%', (x, y), textcoords="offset points",
                    xytext=(0, 12), ha='center', fontsize=9)

    ax.set_xlabel('Remaining Parameters (non-zero)', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Impact of Pruning on Test Accuracy', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)



    ax.invert_xaxis()  # more params on left, fewer on right
    plt.tight_layout()
    plt.savefig("plots/pruning_accuracy_vs_params.png", dpi=150)
    plt.close()












    # Test accuracy vs sparsity ──
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(sparsities, test_accs, 'bo-', linewidth=2, markersize=8,
            label='After fine-tuning')
    ax.plot(sparsities, test_accs_before, 'rs--', linewidth=1.5, markersize=6,
            label='Before fine-tuning', alpha=0.7)
    ax.set_xlabel('Sparsity (%)', fontsize=12)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Test Accuracy vs. Pruning Sparsity', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/pruning_accuracy_vs_sparsity.png", dpi=150)
    plt.close()







    # FLOPs reduction ──
    fig, ax = plt.subplots(figsize=(10, 6))
    baseline_flops = flops[0]
    flops_reduction = [100 * (1 - f / baseline_flops) for f in flops]
    ax.plot(sparsities, flops_reduction, 'g^-', linewidth=2, markersize=8)
    ax.set_xlabel('Sparsity (%)', fontsize=12)
    ax.set_ylabel('FLOPs Reduction (%)', fontsize=12)
    ax.set_title('Computational Savings from Pruning', fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("plots/pruning_flops_reduction.png", dpi=150)
    plt.close()




In [14]:


# Print summary table





def print_summary_table(results):
    print(f"\n{'='*95}")
    print(f"{'Sparsity':>10} | {'Non-zero Params':>16} | {'FLOPs':>14} | "
          f"{'Test (no FT)':>13} | {'Test (FT)':>10} | {'Train (FT)':>11}")
    print(f"{'-'*95}")
    for r in results:
        print(f"{r['target_sparsity']*100:>9.0f}% | "
              f"{r['nonzero_params']:>16,} | "
              f"{r['flops']:>14,} | "
              f"{r['test_acc_before_ft']:>12.2f}% | "
              f"{r['test_acc']:>9.2f}% | "
              f"{r['train_acc']:>10.2f}%")
    print(f"{'='*95}")



In [15]:


# Run everything






if __name__ == "__main__":
    #

    MODEL_PATH = "vgg_cifar10_trained.pth"

    if not os.path.exists(MODEL_PATH):
        print(f"ERROR: {MODEL_PATH} not found!")
        exit(1)





    print("Starting pruning experiment...")
    print(f"Model: {MODEL_PATH}")
    print(f"Fine-tune epochs per sparsity level: 5\n")

    results = run_pruning_experiment(
        model_path=MODEL_PATH,
        fine_tune_epochs=5,
    )





    print_summary_table(results)
    plot_results(results)


ERROR: vgg_cifar10_trained.pth not found!
Starting pruning experiment...
Model: vgg_cifar10_trained.pth
Fine-tune epochs per sparsity level: 5


  Sparsity: 0%


FileNotFoundError: [Errno 2] No such file or directory: 'vgg_cifar10_trained.pth'

Starting pruning experiment...
Model: vgg_cifar10_trained.pth
Fine-tune epochs per sparsity level: 5


============================================================
  Sparsity: 0%
============================================================
  [Baseline] Train: 94.38% | Test: 90.68%
  Params: 4,823,111 / 4,823,114 | FLOPs: 419,042,208

============================================================
  Sparsity: 10%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.68%
    Params: 4,341,405 / 4,823,114 (sparsity: 10.0%)
    FLOPs: 403,013,522
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.61% | Time: 18.3s
    Fine-tune epoch 2/5 | Test Acc: 90.88% | Time: 17.7s
    Fine-tune epoch 3/5 | Test Acc: 90.65% | Time: 17.8s
    Fine-tune epoch 4/5 | Test Acc: 90.93% | Time: 17.7s
    Fine-tune epoch 5/5 | Test Acc: 90.82% | Time: 18.0s
  After fine-tuning:
    Train Acc: 94.99% | Test Acc: 90.82%
    Non-zero params: 4,341,405 | FLOPs: 403,013,522

============================================================
  Sparsity: 30%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.68%
    Params: 3,377,988 / 4,823,114 (sparsity: 30.0%)
    FLOPs: 371,419,544
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.67% | Time: 17.8s
    Fine-tune epoch 2/5 | Test Acc: 90.36% | Time: 18.0s
    Fine-tune epoch 3/5 | Test Acc: 90.59% | Time: 17.3s
    Fine-tune epoch 4/5 | Test Acc: 90.72% | Time: 17.6s
    Fine-tune epoch 5/5 | Test Acc: 90.66% | Time: 17.6s
  After fine-tuning:
    Train Acc: 94.92% | Test Acc: 90.66%
    Non-zero params: 3,377,988 | FLOPs: 371,419,544

============================================================
  Sparsity: 50%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.68%
    Params: 2,414,570 / 4,823,114 (sparsity: 49.9%)
    FLOPs: 336,599,346
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.62% | Time: 18.1s
    Fine-tune epoch 2/5 | Test Acc: 90.61% | Time: 17.9s
    Fine-tune epoch 3/5 | Test Acc: 90.68% | Time: 17.5s
    Fine-tune epoch 4/5 | Test Acc: 90.71% | Time: 17.5s
    Fine-tune epoch 5/5 | Test Acc: 90.85% | Time: 17.6s
  After fine-tuning:
    Train Acc: 94.96% | Test Acc: 90.85%
    Non-zero params: 2,414,570 | FLOPs: 336,599,346

============================================================
  Sparsity: 70%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.68%
    Params: 1,451,152 / 4,823,114 (sparsity: 69.9%)
    FLOPs: 286,836,538
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.71% | Time: 18.1s
    Fine-tune epoch 2/5 | Test Acc: 90.88% | Time: 17.9s
    Fine-tune epoch 3/5 | Test Acc: 90.71% | Time: 17.5s
    Fine-tune epoch 4/5 | Test Acc: 90.85% | Time: 18.1s
    Fine-tune epoch 5/5 | Test Acc: 90.81% | Time: 17.9s
  After fine-tuning:
    Train Acc: 94.89% | Test Acc: 90.81%
    Non-zero params: 1,451,152 | FLOPs: 286,836,538

============================================================
  Sparsity: 80%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.69%
    Params: 969,444 / 4,823,114 (sparsity: 79.9%)
    FLOPs: 246,072,362
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.58% | Time: 17.6s
    Fine-tune epoch 2/5 | Test Acc: 90.39% | Time: 17.6s
    Fine-tune epoch 3/5 | Test Acc: 90.85% | Time: 17.5s
    Fine-tune epoch 4/5 | Test Acc: 90.85% | Time: 17.4s
    Fine-tune epoch 5/5 | Test Acc: 90.87% | Time: 17.7s
  After fine-tuning:
    Train Acc: 95.00% | Test Acc: 90.87%
    Non-zero params: 969,444 | FLOPs: 246,072,362

============================================================
  Sparsity: 90%
============================================================
  After pruning (before fine-tune):
    Test Acc: 90.38%
    Params: 487,735 / 4,823,114 (sparsity: 89.9%)
    FLOPs: 158,018,824
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.64% | Time: 17.7s
    Fine-tune epoch 2/5 | Test Acc: 90.76% | Time: 17.4s
    Fine-tune epoch 3/5 | Test Acc: 90.49% | Time: 17.7s
    Fine-tune epoch 4/5 | Test Acc: 90.65% | Time: 17.7s
    Fine-tune epoch 5/5 | Test Acc: 90.53% | Time: 18.1s
  After fine-tuning:
    Train Acc: 94.87% | Test Acc: 90.53%
    Non-zero params: 487,735 | FLOPs: 158,018,824

============================================================
  Sparsity: 95%
============================================================
  After pruning (before fine-tune):
    Test Acc: 86.38%
    Params: 246,880 / 4,823,114 (sparsity: 94.9%)
    FLOPs: 93,493,912
  Fine-tuning for 5 epochs...
    Fine-tune epoch 1/5 | Test Acc: 90.06% | Time: 17.5s
    Fine-tune epoch 2/5 | Test Acc: 90.13% | Time: 17.7s
    Fine-tune epoch 3/5 | Test Acc: 89.96% | Time: 17.7s
    Fine-tune epoch 4/5 | Test Acc: 90.06% | Time: 17.5s
    Fine-tune epoch 5/5 | Test Acc: 89.92% | Time: 17.5s
  After fine-tuning:
    Train Acc: 93.98% | Test Acc: 89.92%
    Non-zero params: 246,880 | FLOPs: 93,493,912

===============================================================================================
  Sparsity |  Non-zero Params |          FLOPs |  Test (no FT) |  Test (FT) |  Train (FT)
-----------------------------------------------------------------------------------------------
        0% |        4,823,111 |    419,042,208 |        90.68% |     90.68% |      94.38%
       10% |        4,341,405 |    403,013,522 |        90.68% |     90.82% |      94.99%
       30% |        3,377,988 |    371,419,544 |        90.68% |     90.66% |      94.92%
       50% |        2,414,570 |    336,599,346 |        90.68% |     90.85% |      94.96%
       70% |        1,451,152 |    286,836,538 |        90.68% |     90.81% |      94.89%
       80% |          969,444 |    246,072,362 |        90.69% |     90.87% |      95.00%
       90% |          487,735 |    158,018,824 |        90.38% |     90.53% |      94.87%
       95% |          246,880 |     93,493,912 |        86.38% |     89.92% |      93.98%
===============================================================================================
Saved: plots/pruning_accuracy_vs_params.png
Saved: plots/pruning_accuracy_vs_sparsity.png
Saved: plots/pruning_flops_reduction.png

Done! Check the plots/ directory for visualizations.
Use the summary table above for your report's metrics table.

